# 🧠 EduForge: Pedagogical Policy Cloning (SFT)
This notebook trains the `unsloth/gemma-2b-it` model on pedagogical trajectories generated by the EduForge DQN agent. It fulfills the OpenEnv requirement for a reproducible training script.

### 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

### 2. Upload Training Data
Please upload the `training_samples.json` file generated by the EduForge environment to the root of this Colab session before proceeding.

In [ ]:
import os
import json
from datasets import Dataset

if not os.path.exists("training_samples.json"):
    print("⚠️ WARNING: training_samples.json not found! Please upload it using the Colab file browser.")
else:
    print("✅ Training data found.")

### 3. Initialize Unsloth & Format Data

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2b-it",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
def format_conversations(examples):
    texts = []
    for conv in examples["messages"]:
        text = ""
        for msg in conv:
            role = msg["role"]
            content = msg["content"]
            if role == "system":
                text += f"<bos><start_of_turn>model\n{content}<end_of_turn>\n"
            elif role == "user":
                text += f"<start_of_turn>user\n{content}<end_of_turn>\n"
            elif role == "assistant":
                text += f"<start_of_turn>model\n{content}<end_of_turn>\n"
        text += "<eos>"
        texts.append(text)
    return {"text": texts}

with open("training_samples.json", "r", encoding="utf-8") as f:
    data = json.load(f)

conversations = []
for sample in data:
    prompt = f"Student State: Confusion {sample['state']['confusion']:.1f}, Attention {sample['state']['attention']:.1f}, Misconception: {sample['state']['misconception']}\n"
    prompt += f"Selected Action: {sample['action']}"
    response = f"Proceeding with {sample['action']} to address {sample['state']['misconception']}."
    
    conversations.append({
        "messages": [
            {"role": "system", "content": "You are EduForge, an intelligent tutoring agent."},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response}
        ]
    })

dataset = Dataset.from_list(conversations)
dataset = dataset.map(format_conversations, batched=True)
print(f"Loaded {len(dataset)} examples!")

### 4. Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="eduforge_outputs",
    ),
)

trainer_stats = trainer.train()

### 5. Save Checkpoints

In [ ]:
model.save_pretrained("eduforge_lora_model")
tokenizer.save_pretrained("eduforge_lora_model")
print("Model saved successfully!")